In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

DATA_DIR = Path("../data/full_dedup/final evalution")

gold = pd.read_csv(
    DATA_DIR / "manual_gold_pairs_reviewed.csv"
)

gold = gold[
    gold["manual_label"] != "unsure"
].copy()

gold["label"] = (
    gold["manual_label"]
    .astype(int)
)

print(gold.shape)

gold.head()

(300, 10)


,label,left_country,right_country,left_party_name,right_party_name,left_name_latin,right_name_latin,left_cluster_id,right_cluster_id,manual_label
0,0,arm,arm,Ալբերտ Նալբանդյան,Ալբերտ Մեջլումյան,albert nalbandyan,albert mejloumyan,21741,334,0
1,1,arm,arm,Ադել Մուհամեդ Ալի Ջասիմ Ալ Մարզուգի,Ադել Մոհամեդ Ալի Ջասիմ Ալմարզուքի,adel mouhamed ali jasim al marzougi,adel mohamed ali jasim almarzouqi,14805,8164,1
2,0,arm,arm,Աիդա Միրզոյան,ԱԻԴԱ ՀԱՄԲԱՐՁՈՒՄՅԱՆ,aida mirzoyan,aida hambardzoumyan,7686,3048,0
3,1,mng,mng,Лхамсүрэн Уянга,лхамсүрэн уянга,lhamsuren uyanga,lhamsuren uyanga,1299374,1299374,1
4,0,arm,arm,«ԱԼՎԱՆԴ ՄԱՅՆԻՆԳ ԵՎ ՄԻՆԵՐԱԼ ԻՆԴԱՍՏՐԻԶ»,Ալվարդ Սարգսյան,alvand mayning ev mineral indastriz,alvard sargsyan,6489,30582,0


In [3]:
DATA_DIR = Path("../data/full_dedup/final_datasets")
deduplicated = pd.read_parquet(
    DATA_DIR / "deduplicated_records.parquet"
)

records = deduplicated.copy()

records = records.reset_index().rename(
    columns={"index": "record_idx"}
)

records = records.set_index("record_idx")

print(records.shape)

(3653581, 18)


In [4]:
gold.head()

,label,left_country,right_country,left_party_name,right_party_name,left_name_latin,right_name_latin,left_cluster_id,right_cluster_id,manual_label
0,0,arm,arm,Ալբերտ Նալբանդյան,Ալբերտ Մեջլումյան,albert nalbandyan,albert mejloumyan,21741,334,0
1,1,arm,arm,Ադել Մուհամեդ Ալի Ջասիմ Ալ Մարզուգի,Ադել Մոհամեդ Ալի Ջասիմ Ալմարզուքի,adel mouhamed ali jasim al marzougi,adel mohamed ali jasim almarzouqi,14805,8164,1
2,0,arm,arm,Աիդա Միրզոյան,ԱԻԴԱ ՀԱՄԲԱՐՁՈՒՄՅԱՆ,aida mirzoyan,aida hambardzoumyan,7686,3048,0
3,1,mng,mng,Лхамсүрэн Уянга,лхамсүрэн уянга,lhamsuren uyanga,lhamsuren uyanga,1299374,1299374,1
4,0,arm,arm,«ԱԼՎԱՆԴ ՄԱՅՆԻՆԳ ԵՎ ՄԻՆԵՐԱԼ ԻՆԴԱՍՏՐԻԶ»,Ալվարդ Սարգսյան,alvand mayning ev mineral indastriz,alvard sargsyan,6489,30582,0


In [5]:
DATA_DIR = Path("../data/full_dedup/final evalution")
gold = pd.read_csv(DATA_DIR / "manual_gold_pairs_reviewed.csv")

gold = gold[
    gold["manual_label"].isin([0, 1, "0", "1"])
].copy()

gold["label"] = gold["manual_label"].astype(int)

print(gold.shape)
print(gold["label"].value_counts())

(300, 10)
label
1    163
0    137
Name: count, dtype: int64


In [6]:
from difflib import SequenceMatcher

def simple_features(row):
    left = str(row["left_name_latin"])
    right = str(row["right_name_latin"])

    left_tokens = set(left.split())
    right_tokens = set(right.split())

    token_intersection = len(left_tokens & right_tokens)
    token_union = len(left_tokens | right_tokens)

    return {
        "name_similarity": SequenceMatcher(None, left, right).ratio(),
        "token_jaccard": token_intersection / token_union if token_union else 0,
        "same_country": int(row["left_country"] == row["right_country"]),
        "same_cluster": int(row["left_cluster_id"] == row["right_cluster_id"]),
        "len_diff": abs(len(left) - len(right)),
    }

In [7]:
eval_features = pd.DataFrame(
    [simple_features(row) for _, row in gold.iterrows()]
)

y_true = gold["label"]

eval_features.head()

,name_similarity,token_jaccard,same_country,same_cluster,len_diff
0,0.647059,0.333333,1,0,0
1,0.941176,0.375000,1,0,2
2,0.750000,0.333333,1,0,6
3,1.000000,1.000000,1,1,0
4,0.360000,0.000000,1,0,20


In [8]:
y_pred_cluster = eval_features["same_cluster"]

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("Precision:", precision_score(y_true, y_pred_cluster))
print("Recall:", recall_score(y_true, y_pred_cluster))
print("F1:", f1_score(y_true, y_pred_cluster))

print(confusion_matrix(y_true, y_pred_cluster))
print(classification_report(y_true, y_pred_cluster))

Precision: 0.9733333333333334
Recall: 0.8957055214723927
F1: 0.9329073482428115
[[133   4]
 [ 17 146]]
              precision    recall  f1-score   support

           0       0.89      0.97      0.93       137
           1       0.97      0.90      0.93       163

    accuracy                           0.93       300
   macro avg       0.93      0.93      0.93       300
weighted avg       0.93      0.93      0.93       300



In [9]:
# ============================================================
# ERROR ANALYSIS
# ============================================================

gold["pred"] = (
    gold["left_cluster_id"]
    == gold["right_cluster_id"]
).astype(int)

gold["error_type"] = "correct"

gold.loc[
    (gold["label"] == 0)
    & (gold["pred"] == 1),
    "error_type"
] = "false_positive"

gold.loc[
    (gold["label"] == 1)
    & (gold["pred"] == 0),
    "error_type"
] = "false_negative"

print(gold["error_type"].value_counts())

error_type
correct           279
false_negative     17
false_positive      4
Name: count, dtype: int64


In [10]:
# ============================================================
# FALSE POSITIVES
# ============================================================

false_positives = gold[
    gold["error_type"] == "false_positive"
].copy()

print("False positives:", len(false_positives))

false_positives[
    [
        "left_party_name",
        "right_party_name",
        "left_name_latin",
        "right_name_latin",
        "left_country",
        "right_country",
        "left_cluster_id",
        "right_cluster_id",
    ]
]

False positives: 4


,left_party_name,right_party_name,left_name_latin,right_name_latin,left_country,right_country,left_cluster_id,right_cluster_id
62,Мөнхбат Аюуш,Аюуш Мөнхбат,monhbat ayuush,ayuush monhbat,mng,mng,1289695,1289695
85,цэгмид мягмар,цэгмид мягмар,tsegmid myagmar,tsegmid myagmar,mng,mng,1350175,1350175
227,Баттөмөр Одонгэрэл,Одонгэрэл Баттөмөр,battomor odongerel,odongerel battomor,mng,mng,1339872,1339872
261,Азжаргал Бямбаа,Азжаргал Нямаа,azzhargal byambaa,azzhargal nyamaa,mng,mng,1401285,1401285


In [11]:
# ============================================================
# FALSE NEGATIVES
# ============================================================

false_negatives = gold[
    gold["error_type"] == "false_negative"
].copy()

print("False negatives:", len(false_negatives))

false_negatives[
    [
        "left_party_name",
        "right_party_name",
        "left_name_latin",
        "right_name_latin",
        "left_country",
        "right_country",
        "left_cluster_id",
        "right_cluster_id",
    ]
].head(50)

False negatives: 17


,left_party_name,right_party_name,left_name_latin,right_name_latin,left_country,right_country,left_cluster_id,right_cluster_id
1,Ադել Մուհամեդ Ալի Ջասիմ Ալ Մարզուգի,Ադել Մոհամեդ Ալի Ջասիմ Ալմարզուքի,adel mouhamed ali jasim al marzougi,adel mohamed ali jasim almarzouqi,arm,arm,14805,8164
5,Ա.Տ.Ս. Նոմինիս Լիմիթիդ (Կիպրոս),Ա.Տ.Ս. Նոմինիս Լիմիթիդ (Կիպրոս),a t s nominis limitid kipros,a t s nominis limitid kipros,arm,arm,108556,175804
64,Ադել Մուհամմադ Ալի Ջասիմ Ալ Մարզուqի,Ադել Մոհամեդ Ալի Ջասիմ Ալմարզուքի,adel mouhammad ali jasim al marzouqi,adel mohamed ali jasim almarzouqi,arm,arm,8165,8164
80,Ալեքսանդր Տրունով,Ալեքսանդր Տրունով,aleqsandr trounov,aleqsandr trounov,arm,arm,66641,23021
81,Ալեքսանդր Տրունով,Ալեքսանդր Վորոնկով,aleqsandr trounov,aleqsandr voronkov,arm,arm,169778,10268
108,Ադել Մուհամմադ Ալի Ջասիմ Ալ Մարզուqի,Ադել Մոհամեդ Ալի Ջասիմ Ալմարզուքի,adel mouhammad ali jasim al marzouqi,adel mohamed ali jasim almarzouqi,arm,arm,8165,8164
113,Ա.Տ.Ս. Նոմինիս Լիմիթիդ (Կիպրոս),Ա.Տ.Ս. Նոմինիս Լիմիթիդ (Կիպրոս),a t s nominis limitid kipros,a t s nominis limitid kipros,arm,arm,76867,135469
121,Ա.Տ.Ս. Նոմինիս Լիմիթիդ (Կիպրոս),Ա.Տ.Ս. Նոմինիս Լիմիթիդ (Կիպրոս),a t s nominis limitid kipros,a t s nominis limitid kipros,arm,arm,94811,108893
155,Ա.Տ.Ս. Նոմինիս Լիմիթիդ (Կիպրոս),Ա.Տ.Ս. Նոմինիս Լիմիթիդ (Կիպրոս),a t s nominis limitid kipros,a t s nominis limitid kipros,arm,arm,175804,103232
178,Աբրահամ Մելքոնյան,Աբրահամ Զաքարյան,abraham melqonyan,abraham zaqaryan,arm,arm,7988,3361


In [12]:
# ============================================================
# ERROR CATEGORY ANALYSIS
# ============================================================

def classify_error(row):

    left = str(row["left_name_latin"])
    right = str(row["right_name_latin"])

    left_tokens = set(left.split())
    right_tokens = set(right.split())

    overlap = len(left_tokens & right_tokens)

    sim = SequenceMatcher(
        None,
        left,
        right
    ).ratio()

    # --------------------------------------------------------
    # exact / near exact
    # --------------------------------------------------------

    if sim > 0.95:
        return "near_exact_match"

    # --------------------------------------------------------
    # token order swap
    # --------------------------------------------------------

    if (
        left_tokens == right_tokens
        and left != right
    ):
        return "token_order_variation"

    # --------------------------------------------------------
    # transliteration variation
    # --------------------------------------------------------

    if sim > 0.85:
        return "transliteration_variation"

    # --------------------------------------------------------
    # same first name only
    # --------------------------------------------------------

    left_first = left.split()[0] if left.split() else ""
    right_first = right.split()[0] if right.split() else ""

    if left_first == right_first:
        return "same_first_name"

    # --------------------------------------------------------
    # low similarity
    # --------------------------------------------------------

    return "other"


false_negatives["error_category"] = (
    false_negatives
    .apply(classify_error, axis=1)
)

false_positives["error_category"] = (
    false_positives
    .apply(classify_error, axis=1)
)

print("=" * 60)
print("FALSE NEGATIVE CATEGORIES")
print("=" * 60)

print(
    false_negatives["error_category"]
    .value_counts()
)

print()

print("=" * 60)
print("FALSE POSITIVE CATEGORIES")
print("=" * 60)

print(
    false_positives["error_category"]
    .value_counts()
)

FALSE NEGATIVE CATEGORIES
error_category
near_exact_match             11
transliteration_variation     4
same_first_name               2
Name: count, dtype: int64

FALSE POSITIVE CATEGORIES
error_category
token_order_variation        2
near_exact_match             1
transliteration_variation    1
Name: count, dtype: int64


In [13]:
# ============================================================
# FINAL ERROR ANALYSIS SUMMARY
# ============================================================

summary = {
    "total_pairs": len(gold),
    "false_positives": len(false_positives),
    "false_negatives": len(false_negatives),
    "fp_rate": len(false_positives) / len(gold),
    "fn_rate": len(false_negatives) / len(gold),
}

summary

{'total_pairs': 300,
 'false_positives': 4,
 'false_negatives': 17,
 'fp_rate': 0.013333333333333334,
 'fn_rate': 0.056666666666666664}

**Вывод:** По результатам ручной проверки система показала высокое качество дедупликации: итоговый F1-score составил 0.933 при Precision 0.973 и Recall 0.896. Это означает, что модель практически не объединяет разные сущности ошибочно и при этом успешно находит большинство реальных дубликатов.

Основная часть ошибок относится к false negatives — случаи, когда реальные дубликаты не были объединены в один кластер. Большинство таких ошибок связано с:

вариациями транслитерации (mohamed / mouhamed, marzougi / marzouqi);
near-exact multilingual matches;
ограничениями blocking strategy, из-за которых некоторые пары не попадали на этап сравнения.

False positives встречались редко и в основном были вызваны перестановкой токенов в монгольских именах (name surname ↔ surname name).

В целом результаты показывают, что разработанная multilingual graph-based entity resolution pipeline обеспечивает устойчивую и масштабируемую дедупликацию записей даже на шумных многоязычных данных.